# LegalAI — Model Training & Evaluation Notebook

This notebook covers:
1. Embedding model evaluation
2. Section prediction accuracy testing
3. Summarization quality assessment
4. RAG pipeline evaluation

In [ ]:
import sys
sys.path.append('../backend')

from app.services.embeddings import EmbeddingService
from app.services.section_predictor import section_predictor
from app.services.legal_parser import legal_parser
from app.database.chroma_db import chroma_manager

print('All imports successful!')

## 1. Test Embedding Service

In [ ]:
import asyncio

embedding_service = EmbeddingService()

# Test single text embedding
text = "The accused was charged under Section 302 of IPC for murder"
embedding = await embedding_service.embed_text(text)
print(f'Embedding dimension: {len(embedding)}')
print(f'First 5 values: {embedding[:5]}')

# Test batch embeddings
texts = [
    "Murder conviction under Section 302 IPC",
    "Cyber fraud under Section 66 IT Act",
    "Theft of property under Section 379 IPC",
]
embeddings = await embedding_service.embed_texts(texts)
print(f'\nBatch embeddings: {len(embeddings)} vectors of dimension {len(embeddings[0])}')

## 2. Test Section Prediction

In [ ]:
# Test with sample legal text
sample_text = """
The accused, Raj Kumar, was charged under Section 302 of the Indian Penal Code 
for the murder of Sunita Devi on 15th March 2023. The prosecution presented 
DNA evidence and CCTV footage. The court also considered Section 376 IPC for rape 
and Section 201 IPC for causing disappearance of evidence. The accused was convicted 
and sentenced to life imprisonment.
"""

result = section_predictor.predict_sections(sample_text, "test_doc_1")

print(f'Total sections found: {result.total_sections_found}')
print(f'Acts referenced: {[a.value for a in result.acts_referenced]}')
print()

for ps in result.predicted_sections:
    print(f'[{ps.confidence:.0%}] {ps.section.act} Section {ps.section.section_number}: {ps.section.section_title}')
    print(f'  Reason: {ps.reason[:100]}...')
    print()

## 3. Test Legal Parser

In [ ]:
# Test entity extraction
metadata, entities, doc_type = legal_parser.parse(sample_text)

print(f'Document Type: {doc_type.value}')
print(f'Case Number: {metadata.case_number}')
print(f'Court: {metadata.court_name}')
print(f'Crime Category: {metadata.crime_category}')
print(f'Parties: {metadata.parties}')
print(f'Keywords: {metadata.keywords}')
print(f'\nTotal entities: {len(entities)}')

for entity in entities:
    print(f'  [{entity.entity_type}] {entity.value} ({entity.confidence:.0%})')

## 4. Test Section Explanation

In [ ]:
# Test section explanation
explanation = section_predictor.explain_section('IPC', '302')

if explanation:
    print(f'Section: {explanation["section_title"]}')
    print(f'\nSimple Explanation:')
    print(explanation['simple_explanation'])
    print(f'\nLegal Implications:')
    print(explanation['legal_implications'])
    print(f'\nExample Cases:')
    for case in explanation['example_cases']:
        print(f'  • {case}')

## 5. Evaluation Metrics

Evaluate section prediction against a labeled test set.

In [ ]:
# Test cases with known sections
test_cases = [
    {
        "text": "The accused committed murder under Section 302 IPC",
        "expected_sections": ["302"],
        "expected_acts": ["Indian Penal Code"],
    },
    {
        "text": "Cyber fraud and identity theft under Section 66 and 66C of IT Act",
        "expected_sections": ["66", "66C"],
        "expected_acts": ["Information Technology Act"],
    },
    {
        "text": "Dowry death case under Section 304B IPC and cruelty under 498A",
        "expected_sections": ["304B", "498A"],
        "expected_acts": ["Indian Penal Code"],
    },
]

precision_scores = []
recall_scores = []

for test in test_cases:
    result = section_predictor.predict_sections(test['text'], 'eval')
    predicted = {ps.section.section_number for ps in result.predicted_sections}
    expected = set(test['expected_sections'])
    
    if len(predicted) > 0:
        precision = len(predicted & expected) / len(predicted)
        precision_scores.append(precision)
    if len(expected) > 0:
        recall = len(predicted & expected) / len(expected)
        recall_scores.append(recall)
    
    print(f'Text: {test["text"][:60]}...')
    print(f'Expected: {expected}')
    print(f'Predicted: {predicted}')
    print(f'Precision: {precision:.2f}, Recall: {recall:.2f}')
    print()

print(f'\nOverall Precision: {sum(precision_scores)/len(precision_scores):.2f}')
print(f'Overall Recall: {sum(recall_scores)/len(recall_scores):.2f}')